# GSB 5544 — PA 4.2: Decode a Message — SOLUTION

## Setup

Run the code below to load the scrambled message:


In [1]:
import pandas as pd

message = pd.read_csv("https://www.dropbox.com/s/lgpn3vmksk3ssdo/scrambled_message.txt?dl=1")['Word']

In [2]:
message

0                    Koila!
1                     In   
2                     kiew,
3                         a
4                 humble   
               ...         
122                     you
123                 mabugh.
124              ughhh?call
125        meugh.ughhhh!   
126                      K.
Name: Word, Length: 127, dtype: str

In this activity, a "word" refers to any set of characters with no white space, even though they are not truly an English word.  That is, even though many of elements of the scrambled message vector are nonsense, and some have punctuation, you can consider each element to be a "word".

Beware!  The object named `message` is a **pandas Series** of strings. You might want to join the series of strings together into a single string. If you want to use functions that expect a string, rather than a series, you will need `.apply()` and `lambda` functions.




## Warm-up exercises

1. How many characters are in the scrambled message?
2. How many of these characters are white space?
3. How many words are in the scrambled message?
4. Show all the punctuation marks in the scrambled message.
5. Print out, in all capitals, the longest word in the scrambled message.
6. Print out every piece of a word that starts with the letter "m" and ends with the letter "z" in the scrambled message.

In [3]:
import re

# 1. How many characters are in the scrambled message?
message.str.len().sum()

np.int64(2544)

In [4]:
# 2. How many of these characters are white space?
message.str.count(r"\s").sum()

np.int64(1652)

In [5]:
# 3. How many words are in the scrambled message?  (one "word" per element)
len(message)

127

In [6]:
# 4. Show all the punctuation marks in the scrambled message.
punctuation = message.apply(lambda w: re.findall(r"[^\w\s]", w))
all_punct = [p for lst in punctuation for p in lst]
print(all_punct)
print("distinct marks:", sorted(set(all_punct)))

['!', ',', '?', ',', '!', '!', '.', '!', ',', '!', '?', '?', '?', '?', '?', ',', '?', '!', ',', '.', '?', ',', ',', '!', '.', '!', '?', '!', '!', ',', '.', '?', '?', '?', '.', '?', '?', '.', '.', ';', '?', ',', '!', '!', ',', '.', '?', '.', '?', '.', '!', '!', '.', '!', '!', '?', ',', '!', '?', '.', '!', '!', '!', '.', '?', '.', '!', '.']
distinct marks: ['!', ',', '.', ';', '?']


In [7]:
# 5. Print out, in all capitals, the longest word in the scrambled message.
trimmed = message.str.strip()
longest = trimmed[trimmed.str.len().idxmax()]
print(longest.upper())

KAUDEVILLIANUGH?AOGHAJDBN


In [8]:
# 6. Every piece of a word that starts with "m" and ends with "z".
pieces = message.apply(lambda w: re.findall(r"m\w*z", w))
[p for lst in pieces for p in lst]

['mosz', 'maaz']

**Answer:** 2,544 characters in total, of which 1,652 are whitespace (the words come padded with
spaces); 127 words. The punctuation is limited to `! , . ; ?`. The longest word is
`KAUDEVILLIANUGH?AOGHAJDBN` (25 characters), and the m…z pieces are `mosz` and `maaz` —
which, once decoded (z → t, aa → ee), will become *most* and *meet*.



## Decode a message

Complete the following steps to decode the message.  

1. Remove any spaces before or after each word.
2. Any time you see the word "ugh", with any number of h's, followed by a punctuation mark, delete this.
3. No word should be longer than 16 characters. Drop all extra characters beyond 13 off the end of each word.
4. Replace all instances of exactly 2 a's with exactly 2 e's.
5. Replace all z's with t's.
6. Every word that ends in b, change that to a y.  *Hint: look out for punctuation!*
7. Every word that starts with k, change that to a v.  *Hint: look out for capitalization!*
8. Use `.join()` to recombine all your words into a message.
9. Find the movie this quote is from.

In [9]:
# 1. Remove any spaces before or after each word.
words = message.str.strip()

# 2. "ugh" with any number of h's, followed by a punctuation mark: delete it.
words = words.str.replace(r"ugh+[^\w\s]", "", regex=True)

# 3. No word should be longer than 16 characters: drop the extras off the end.
words = words.str.slice(0, 16)

# 4. Replace all instances of exactly 2 a's with exactly 2 e's.
words = words.str.replace("aa", "ee")            # literal text -> no regex needed

# 5. Replace all z's with t's.
words = words.str.replace("z", "t")

# 6. Every word that ends in b: change it to a y.  (The b may hide before punctuation!)
words = words.str.replace(r"b([^\w\s]*)$", r"y\1", regex=True)

# 7. Every word that starts with k: change it to a v.  (Mind the capital K's!)
words = words.str.replace(r"^k", "v", regex=True).str.replace(r"^K", "V", regex=True)

words

0      Voila!
1          In
2       view,
3           a
4      humble
        ...  
122       you
123       may
124      call
125        me
126        V.
Name: Word, Length: 127, dtype: str

In [10]:
# 8. Recombine the words into a message.
decoded = " ".join(words)
print(decoded)

Voila! In view, a humble vaudevillianaogh veteran, cast vicariously as both victim and villain by the vicissitudes of fate. This visage, no mere veneer of vanity, is a vestige of the vox populi now vacant, vanished. However, this valorous visitation of a bygone vexation stands vivified, and has vowed to vanquish these venal and virulent vermin, van guarding vice and vouchsafing the violently vicious and voracious violation of volition. The only verdict is vengeance; a vendetta, held as a votive not in vain, for the value and veracity of such shall one day vindicate the vigilant and the virtuous. Verily this vichyssoise of verbiage veers most verbose, so let me simply add that its my very good honour to meet you and you may call me V.


**Answer:** > *"Voila! In view, a humble vaudevillian veteran, cast vicariously as both victim and villain
> by the vicissitudes of fate. … The only verdict is vengeance; a vendetta … and you may call me V."*

**9.** The quote is V's alliterative introduction speech from ***V for Vendetta*** (2005).

Notes on the trickier steps: in step 2 the pattern `ugh+[^\w\s]` needs the *punctuation
class* after the h's — `ugh` also hides inside legitimate words (`kaudevillian…` →
*vaudevillian*), and only the groans are followed by punctuation. Step 6's `b([^\w\s]*)$`
keeps any trailing punctuation while swapping the letter (a bare `b$` would miss `b,`).
Step 7 needs both `^k` and `^K` (regex is case-sensitive — `Koila!` must become `Voila!`).
And step 3's instructions mention both 16 and 13; we keep 16 characters, matching "no word
should be longer than 16."